In [0]:
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI

# Authenticate using your Databricks workspace session
workspace_client = WorkspaceClient()

# Create a client for Unity Gateway
client = DatabricksOpenAI(
    workspace_client=workspace_client,
    use_ai_gateway=True
)

MODEL = "course_ops.gateway.chat"

print("Client initialized successfully!")

In [0]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an MLOps expert who helps data "
                "scientists investigate production ML incidents."
            )
        },
        {
            "role": "user",
            "content": (
                "A purchase propensity model's average prediction "
                "score suddenly dropped by 50%. "
                "What could be causing this, and what would "
                "you investigate first?"
            )
        }
    ],
    reasoning_effort="low",
    max_tokens=1024
)

print(response.choices[0].message.content)

In [0]:
usage = response.usage

if usage is not None:
    print("Input tokens:", usage.prompt_tokens)
    print("Output tokens:", usage.completion_tokens)
    print("Total tokens:", usage.total_tokens)
else:
    print("Token usage not reported by the service.")

In [0]:
prompts = [
    # Experiment 1: General diagnosis
    """
    A purchase propensity model's average prediction
    score dropped by 50% overnight.

    What are the three most important things
    you would investigate first?
    """,

    # Experiment 2: Additional monitoring evidence
    """
    A purchase propensity model shows:
    - Previous average prediction score: 0.115
    - Current average prediction score: 0.053
    - Previous feature null rate: 2%
    - Current feature null rate: 55%
    - Pipeline status: SUCCESS

    What can you conclude from this information?
    What additional evidence would you need?
    """,

    # Experiment 3: Insufficient information
    """
    A data scientist asks:
    'Why did our model predictions change yesterday?'

    You have no monitoring metrics, logs,
    or historical data available.

    How should you respond?
    """
]

results = []

for i, prompt in enumerate(prompts, start=1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an MLOps incident investigator. "
                    "Distinguish observed facts from hypotheses. "
                    "Never invent evidence."
                )
            },
            {"role": "user", "content": prompt}
        ],
        reasoning_effort="low",
        max_tokens=1024
    )

    answer = response.choices[0].message.content
    usage = response.usage

    results.append({
        "experiment": i,
        "prompt": prompt.strip(),
        "answer": answer,
        "input_tokens": usage.prompt_tokens if usage else None,
        "output_tokens": usage.completion_tokens if usage else None,
        "total_tokens": usage.total_tokens if usage else None
    })

    print(f"\n{'=' * 50}")
    print(f"EXPERIMENT {i}")
    print(f"{'=' * 50}")
    print(answer)
    print("\nToken usage:", results[-1]["total_tokens"])